In [1]:
!pip uninstall -y protobuf
!pip install protobuf==3.20.3

Found existing installation: protobuf 5.29.5
Uninstalling protobuf-5.29.5:
  Successfully uninstalled protobuf-5.29.5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.1/162.1 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
a2a-sdk 0.3.22 requires protobuf>=5.29.5, but you have protobuf 3.20.3 which is incompatible.
onnx 1.20.1 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 3.20.3 which is incompatible.
tensorflow-metadata 1.17.2 requires protobuf>=4.25.2; python_version >= "3.11", but you have protobuf 3.20.3 which is incompatible.
ydf 0.13.0 requir

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [3]:
%env TOKENIZERS_PARALLELISM=false

env: TOKENIZERS_PARALLELISM=false


In [4]:
import io
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pickle
import re
import seaborn as sns
import tokenize
import torch
import torch.nn as nn
import torch.optim as optim
import transformers

from datasets import Dataset

from math import ceil

from scipy.special import softmax

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
from sklearn.model_selection import StratifiedShuffleSplit

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    logging
)

from torch.utils.data import DataLoader

from tqdm.auto import tqdm

2026-01-19 00:12:15.700908: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768781535.891031      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768781535.946485      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1768781536.404987      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768781536.405031      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1768781536.405034      55 computation_placer.cc:177] computation placer alr

In [5]:
path = "/kaggle/working/state.db"

if os.path.exists(path):
    os.remove(path)
    print("state.db deleted")
else:
    print("state.db not found")


state.db not found


In [6]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    macro_f1 = f1_score(labels, predictions, average="macro")
    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average="macro")
    recall = recall_score(labels, predictions, average="macro")

    return {
        "macro_f1": macro_f1,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall
    }

In [7]:
base_path = "/kaggle/input/sem-eval-2026-task-13-subtask-b/Task_B"

training_path = base_path + "/train.parquet"
validation_path = base_path + "/validation.parquet"
test_path = base_path + "/test.parquet"

training_df = pd.read_parquet(training_path)
validation_df = pd.read_parquet(validation_path)
test_df = pd.read_parquet(test_path)

In [8]:
llm_df = training_df[training_df["generator"].str.lower() != "human"]
human_df = training_df[training_df["generator"].str.lower() == "human"]

TARGET_HUMAN_RATIO = 0.65

num_llm = len(llm_df)
num_human = int(num_llm * TARGET_HUMAN_RATIO / (1 - TARGET_HUMAN_RATIO))

human_df = human_df.sample(
    n=min(len(human_df), num_human),
    random_state=1337
)

training_df = pd.concat([llm_df, human_df]).sample(frac=1, random_state=1337)

In [9]:
training_df.to_parquet("training_sample_set.parquet", index=False)
validation_df.to_parquet("validation_sample_set.parquet", index=False)

In [10]:
pretrained_model = "microsoft/unixcoder-base"

In [11]:
tokenizer = AutoTokenizer.from_pretrained(pretrained_model)

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

In [12]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [13]:
counts = training_df["label"].value_counts().sort_index()

weights = 1.0 / torch.sqrt(
    torch.tensor(counts.values, dtype=torch.float)
)
weights = weights / weights.mean()

class_weights = weights.to(device)

In [14]:
label_counts = training_df["label"].value_counts().sort_index()

class_priors = label_counts / label_counts.sum()
class_priors = class_priors.values.astype(np.float32)

class_priors = class_priors / class_priors.sum()

In [15]:
with open("class_priors.pkl", "wb") as f:
    pickle.dump(class_priors, f)

In [16]:
def preprocess_function(examples: pd.DataFrame):
    tokenized = tokenizer(
        examples["code"],
        truncation=True,
        max_length=192
    )

    return tokenized

In [17]:
training_dataset = Dataset.from_pandas(training_df)
validation_dataset = Dataset.from_pandas(validation_df)

training_tokenized_set = training_dataset.map(preprocess_function, batched=True)
validation_tokenized_set = validation_dataset.map(preprocess_function, batched=True)

training_tokenized_set.set_format("torch")
validation_tokenized_set.set_format("torch")

Map:   0%|          | 0/165440 [00:00<?, ? examples/s]

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

In [18]:
id2label = {
  "0": "human",
  "1": "deepseek",
  "2": "qwen",
  "3": "01-ai",
  "4": "bigcode",
  "5": "gemma",
  "6": "phi",
  "7": "meta-llama",
  "8": "ibm-granite",
  "9": "mistral",
  "10": "openai"
}


label2id = {
  "human": "0",
  "deepseek": "1",
  "qwen": "2",
  "01-ai": "3",
  "bigcode": "4",
  "gemma": "5",
  "phi": "6",
  "meta-llama": "7",
  "ibm-granite": "8",
  "mistral": "9",
  "openai": "10"
}

In [19]:
model = AutoModelForSequenceClassification.from_pretrained(
    pretrained_model,
    num_labels=11,
    id2label=id2label,
    label2id=label2id,
    hidden_dropout_prob=0.2,
    attention_probs_dropout_prob=0.2
)

config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/504M [00:00<?, ?B/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at microsoft/unixcoder-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [20]:
model.to(device)
print("\nRunning on device:", device)

model.safetensors:   0%|          | 0.00/504M [00:00<?, ?B/s]


Running on device: cuda


In [21]:
print(torch.cuda.device_count())
print(torch.cuda.get_device_name(0))

1
Tesla T4


In [22]:
logging.set_verbosity_info()

In [23]:
class TrainingProgressCallback(transformers.TrainerCallback):
    def on_step_end(self, args, state, control, **kwargs):
        if state.is_local_process_zero:
            tqdm.write(f"Step {state.global_step}/{state.max_steps}")

In [24]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/checkpoints_b",
    seed=42,
    learning_rate=2e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.02,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=4,
    label_smoothing_factor=0.0,
    num_train_epochs=4,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=True,
    dataloader_num_workers=0,
    report_to=[]
)

PyTorch: setting up devices


In [25]:
class OptimizedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = torch.nn.CrossEntropyLoss(
            weight=class_weights.to(logits.device)
        )


        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

In [26]:
trainer = OptimizedTrainer(
    model=model,
    args=training_args,
    train_dataset=training_tokenized_set,
    eval_dataset=validation_tokenized_set,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

/tmp/ipykernel_55/92150513.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `OptimizedTrainer.__init__`. Use `processing_class` instead.
  trainer = OptimizedTrainer(
Using auto half precision backend


In [27]:
trainer.train()

The following columns in the Training set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: code, language, __index_level_0__, generator. If code, language, __index_level_0__, generator are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.
***** Running training *****
  Num examples = 165,440
  Num Epochs = 4
  Instantaneous batch size per device = 16
  Total train batch size (w. parallel, distributed & accumulation) = 64
  Gradient Accumulation steps = 4
  Total optimization steps = 10,340
  Number of trainable parameters = 125,938,187


Epoch,Training Loss,Validation Loss,Macro F1,Accuracy,Precision,Recall
1,1.153600,0.688568,0.368019,0.877410,0.326862,0.496042
2,1.017500,0.628595,0.430539,0.889810,0.381917,0.537588
3,0.884000,0.648250,0.432268,0.878230,0.388647,0.545501
4,0.817100,0.652052,0.438021,0.877720,0.390983,0.549581


The following columns in the Evaluation set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: code, language, generator. If code, language, generator are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.

***** Running Evaluation *****
  Num examples = 100000
  Batch size = 32
Saving model checkpoint to /kaggle/working/checkpoints_b/checkpoint-2585
Configuration saved in /kaggle/working/checkpoints_b/checkpoint-2585/config.json
Model weights saved in /kaggle/working/checkpoints_b/checkpoint-2585/model.safetensors
tokenizer config file saved in /kaggle/working/checkpoints_b/checkpoint-2585/tokenizer_config.json
Special tokens file saved in /kaggle/working/checkpoints_b/checkpoint-2585/special_tokens_map.json
Deleting older checkpoint [/kaggle/working/checkpoints_b/checkpoint-12064] due to args.save_total_limit
The following columns in the Evaluation set don't have a corresponding argume

TrainOutput(global_step=10340, training_loss=1.0471855857155539, metrics={'train_runtime': 8210.6345, 'train_samples_per_second': 80.598, 'train_steps_per_second': 1.259, 'total_flos': 6.529891570532352e+16, 'train_loss': 1.0471855857155539, 'epoch': 4.0})

In [28]:
trainer.save_model("/kaggle/working/final_model")
tokenizer.save_pretrained("/kaggle/working/final_tokenizer")

Saving model checkpoint to /kaggle/working/final_model
Configuration saved in /kaggle/working/final_model/config.json
Model weights saved in /kaggle/working/final_model/model.safetensors
tokenizer config file saved in /kaggle/working/final_model/tokenizer_config.json
Special tokens file saved in /kaggle/working/final_model/special_tokens_map.json
tokenizer config file saved in /kaggle/working/final_tokenizer/tokenizer_config.json
Special tokens file saved in /kaggle/working/final_tokenizer/special_tokens_map.json


('/kaggle/working/final_tokenizer/tokenizer_config.json',
 '/kaggle/working/final_tokenizer/special_tokens_map.json',
 '/kaggle/working/final_tokenizer/vocab.json',
 '/kaggle/working/final_tokenizer/merges.txt',
 '/kaggle/working/final_tokenizer/added_tokens.json',
 '/kaggle/working/final_tokenizer/tokenizer.json')

In [29]:
# tokenizer = AutoTokenizer.from_pretrained(
#     "/kaggle/working/final_tokenizer"
# )
# model = AutoModelForSequenceClassification.from_pretrained(
#     "/kaggle/working/final_model"
# )

In [30]:
def tokenize_test_set(examples: pd.DataFrame):
    return tokenizer(
        text_target=examples["code"],
        truncation=True,
        max_length=192
    )

In [31]:
test_dataset = Dataset.from_pandas(test_df)

test_tokenized_set = test_dataset.map(tokenize_test_set, batched=True)

test_tokenized_set.set_format("torch")

Map:   0%|          | 0/500000 [00:00<?, ? examples/s]

In [32]:
test_logits = trainer.predict(test_tokenized_set).predictions

probs = softmax(test_logits, axis=1)

weighted_probs = probs * class_priors

preds = weighted_probs.argmax(axis=1)

The following columns in the test set don't have a corresponding argument in `RobertaForSequenceClassification.forward` and have been ignored: ID, code, __index_level_0__. If ID, code, __index_level_0__ are not expected by `RobertaForSequenceClassification.forward`,  you can safely ignore this message.

***** Running Prediction *****
  Num examples = 500000
  Batch size = 32


In [33]:
pd.DataFrame({
    "ID": test_df["ID"],
    "label": preds
}).to_csv("Predictions.csv", index=False)